# SCF abstraction: SCF kernel


In [1]:
import numpy as np
from numpy.typing import NDArray
from typing import Tuple, List, Literal

from py_mods.src.external.DIRAC_ME import (
    build_S_V_W_T_from_h5,
    get_nuc_charge,
    full_eri_from_h5,
    build_uncontracted_basis_from_h5,
)


from py_mods.src.SCF_4c.types_4c import (
    CS_4c_KU_SCF_Context,
    CS_4c_KU_SCF_Constants,
    CS_4c_KU_SCF_State,
    CS_4c_KU_SCF_Results,
    allocate_CS_4c_KU_SCF_extended_context,
    allocate_CS_4c_KU_SCF_state,
    pack_CS_4c_KU_SCF_results,
)


from py_mods.src.SCF_4c.CS_KU_SCF import (
    initialize_CS_4c_KU_SCF_extended_context,
    initialize_CS_4c_KU_SCF_P_and_E,
    initialize_CS_4c_KU_SCF_state_variable,
    g_matrix_4c,
)


from py_mods.src.SCF_4c.scf_4c_kernels import occupation_4c, eri_classified
from py_mods.src.SCF_4c.utils import validate_CS_4c_KU_SCF_context_input

from py_mods.src.SCF.CSRHF import (
    is_converged,
    update_rhf_acc_hist_size,
    conv_acc_criteria_met,
    print_table_header,
)
from py_mods.src.SCF.scf_kernels import calc_residual_commutator
from py_mods.src.SCF.linalg import sign_convention
from py_mods.src.SCF.scf_kernels import calc_residual_commutator

We will take the original kernel and literally translate it:

In [2]:
def _kuscf_kernel(ctx: CS_4c_KU_SCF_Context) -> CS_4c_KU_SCF_Results:
    validate_CS_4c_KU_SCF_context_input(ctx)

    ext_ctx = allocate_CS_4c_KU_SCF_extended_context(ctx)
    state = allocate_CS_4c_KU_SCF_state(ctx)

    initialize_CS_4c_KU_SCF_extended_context(ctx, ext_ctx)

    initialize_CS_4c_KU_SCF_P_and_E(ctx, state)
    initialize_CS_4c_KU_SCF_state_variable(ext_ctx, state)

    if ctx.verbose:
        print_table_header()

    for iter_idx in range(ctx.max_iter):
        state.iteration += 1

        update_CS_4c_KU_SCF_F_and_r_comp(ctx, ext_ctx, state)
        update_CS_4c_KU_SCF_energy(ext_ctx, state)

        if ctx.verbose:
            print_cycle_data_4c(ctx._convergence_criteria, state)

        # Reused RHF functions (duck-typed)
        state.converged = is_converged(ctx, state) # type: ignore
        if state.converged:
            break

        update_rhf_acc_hist_size(ctx, state) # type: ignore
        state.P_old = state.P.copy()

        update_CS_4c_KU_SCF_F_matrix(ctx, state)

        update_CS_4c_KU_SCF_density(ctx, ext_ctx, state)

        # Here we assume that we cannot set imaginary density to 0.
        # if ctx.theta == 0.0:
        #     state.P.imag = state.C_munu.imag = 0

        state.use_conv_acc = conv_acc_criteria_met(ctx, ext_ctx, state) # type: ignore

    update_CS_4c_KU_SCF_density(ctx, ext_ctx, state)
    state.F_next = state.F

    results = pack_CS_4c_KU_SCF_results(ctx, ext_ctx, state)

    return results

Here we assume that the linear algebra for the commutator for convergence is the same. 


And we see that we are missing the definitions of:
- `update_CS_4c_KU_SCF_F_and_r_comp`
- `update_CS_4c_KU_SCF_energy`
- `print_cycle_data_4c`
- `update_CS_4c_KU_SCF_F_matrix`
- `update_CS_4c_KU_SCF_density` 

So that is what we are going to build.

---
## 1. `update_CS_4c_KU_SCF_F_and_r_comp`
Here we have modified the function to build the Fock matrix using the 4c $G$ matrix:

In [3]:
def update_CS_4c_KU_SCF_F_and_r_comp(
    ctx: CS_4c_KU_SCF_Context,
    ext_ctx: CS_4c_KU_SCF_Constants,
    state: CS_4c_KU_SCF_State,
) -> None:
    G = g_matrix_4c(state.P, ext_ctx.eri_scaled)
    state.F = ext_ctx.H_core + G
    state.r = calc_residual_commutator(state.F, state.P, ctx.S.astype(np.complex128))
    state.error = float(np.linalg.norm(state.r.flatten()))

## 2. `update_CS_4c_KU_SCF_energy`
Here we have modified the function in order to use the trace formula instead of the einsum we previously used

In [4]:
def update_CS_4c_KU_SCF_energy(
    ext_ctx: CS_4c_KU_SCF_Constants,
    state: CS_4c_KU_SCF_State,
) -> None:
    e_scf = np.linalg.trace(state.P @ (ext_ctx.H_core + state.F)) * 0.5
    state.E_SCF =  e_scf
    state.E_diff = state.E_SCF - state.E_prev
    state.E_prev = state.E_SCF

## 3. `print_cycle_data_4c`
This should be pretty much the same

In [5]:
def print_cycle_data_4c(convergence_criteria: str, state: CS_4c_KU_SCF_State) -> None:
    print(
        f"{state.iteration:5}     {state.E_SCF:45.16f}     {state.E_diff:45.16f}     {state.error:8.4E}"
    )

## 4. `update_CS_4c_KU_SCF_F_matrix`

Even though it might be the exact same, in the NR limit, we exploited that the Hamiltonian is real (in the unscaled case) and thus: 

$$
\Braket{C|C} = \sum_{\mu\nu} C_{\mu}^* C_{\nu} = \sum_{\mu\nu} C_{\mu} C_{\nu}
$$

Therefore in the extrapolation we considered:

```python
for i in range(n_guesses):
    for j in range(n_guesses):
        # dot product of flattened arrays
        B_matrix[i, j] = np.dot(residuals[i].ravel(), residuals[j].ravel())
```

However, the DF Hamiltonian is not real, and thus we need to consider the complex conjugate:

```python
for i in range(n_guesses):
    for j in range(n_guesses):
        if theta == 0.0:
            B_matrix[i, j] = np.vdot(residuals[i].ravel(), residuals[j].ravel())
```

But when the Hamiltonian is scaled, it is simply a non hermitian Hamiltonian and we can apply c-normalization:

```python
            else:
            # c-norm metric inner product for complex scaling
            B_matrix[i, j] = np.dot(residuals[i].ravel(), residuals[j].ravel())
```


In [6]:
def update_CS_4c_KU_SCF_F_matrix(
    ctx: CS_4c_KU_SCF_Context,
    state: CS_4c_KU_SCF_State,
) -> None:
    if not state.use_conv_acc:
        F_next = state.F
    else:
        try:
            F_opt, r_opt = calc_diis_extrapolation_4c(
                state.residuals, state.F_guess, ctx.theta
            )
            F_next = F_opt

            if ctx.conv_type == "CROP":
                state.F_guess[-1] = F_opt
                state.residuals[-1] = r_opt
        except np.linalg.LinAlgError:
            if ctx.verbose:
                print(
                    "!!!!!!!!!!!!!!!! CONVERGENCE ACCELERATION CAUSED A SINGULAR MATRIX. REVERTING TO STANDARD SCF !!!!!!!!!!!!!!!"
                )
            state.use_conv_acc = False
            F_next = state.F

    state.F_next = F_next


def calc_diis_extrapolation_4c(
    residuals: List[NDArray[np.complex128]],
    F_guesses: List[NDArray[np.complex128]],
    theta: float,
) -> Tuple[NDArray[np.complex128], NDArray[np.complex128]]:
    n_guesses = len(residuals)
    eq_sis_dim = n_guesses + 1

    B_matrix = np.zeros((eq_sis_dim, eq_sis_dim), dtype=np.complex128)
    B_matrix[-1, :] = 1
    B_matrix[:, -1] = 1
    B_matrix[-1, -1] = 0

    for i in range(n_guesses):
        for j in range(n_guesses):
            if theta == 0.0:
                # We have to use complex conjugation because the DF Hamiltonian is
                # complex. In the case of the NR case we could just do the scalar
                # product since the hamiltonian is real so we could get away
                # with using np.dot in both cases.
                B_matrix[i, j] = np.vdot(residuals[i].ravel(), residuals[j].ravel())
            else:
                # c-norm metric inner product for complex scaling
                B_matrix[i, j] = np.dot(residuals[i].ravel(), residuals[j].ravel())

    solution = np.zeros(eq_sis_dim, dtype=np.complex128)
    solution[-1] = 1

    try:
        c = np.linalg.solve(B_matrix, solution)
    except np.linalg.LinAlgError:
        raise np.linalg.LinAlgError("DIIS matrix singular")

    coeffs = c[:-1]

    F_conv = np.zeros_like(F_guesses[0])
    r_conv = np.zeros_like(residuals[0])

    for k, coef in enumerate(coeffs):
        F_conv += coef * F_guesses[k]
        r_conv += coef * residuals[k]

    return F_conv, r_conv

## 5. `update_CS_4c_KU_SCF_density`

In [7]:
def update_CS_4c_KU_SCF_density(
    ctx: CS_4c_KU_SCF_Context,
    ext_ctx: CS_4c_KU_SCF_Constants,
    state: CS_4c_KU_SCF_State,
) -> None:
    state.P, state.e_orb, state.C_munu, state.C_prime = calculate_P_next_4c(
        state.F_next, ext_ctx.X, ext_ctx.det, ext_ctx._eigensolver, ctx.theta
    )

    state.C_munu = sign_convention(state.C_munu)
    return


def calculate_P_next_4c(
    F_next: NDArray[np.complex128],
    X: NDArray[np.complex128],
    det: NDArray[np.int8],
    solver: Literal["eig", "eigh"],
    theta: float,
) -> Tuple[
    NDArray[np.complex128],
    NDArray[np.complex128],
    NDArray[np.complex128],
    NDArray[np.complex128],
]:
    from py_mods.src.SCF.scf_kernels import calc_p_matrix_comp

    F_prime = X.T @ F_next @ X

    if solver == "eigh":
        e_values, C_prime = np.linalg.eigh(F_prime)
        idx = np.argsort(e_values.real)
        e_values = e_values[idx]
        C_prime = C_prime[:, idx]
    else:
        e_values, C_prime = np.linalg.eig(F_prime)
        idx = np.argsort(e_values.real)
        e_values = e_values[idx]
        C_prime = C_prime[:, idx]

    C_munu = X @ C_prime

    if theta == 0.0:
        L_munu = C_munu.conj().T
    else:
        L_munu = C_munu.T

    P_munu = calc_p_matrix_comp(L_munu, C_munu, det)

    return P_munu, e_values, C_munu, C_prime

# SCF test of unscaled case

In [8]:
h5_filename = "data/He_checkpoint.h5"

S, V, W, T = build_S_V_W_T_from_h5(h5_filename)
H_nuc = V + W + T
nuc_charge = get_nuc_charge(h5_filename)

_, nL, nS = build_uncontracted_basis_from_h5(h5_filename)
eri = full_eri_from_h5(h5_filename)
eri = eri_classified(eri, nL)

occ_det = occupation_4c(nS, nL, 2)

test_ctx = CS_4c_KU_SCF_Context(
    nL,
    nS,
    S,
    T,
    V,
    W,
    eri,
    2,
    theta=0.00,
    occ=occ_det,
    p_guess="ones",
    verbose=True,
    threshold=1e-10,
)

test_const = allocate_CS_4c_KU_SCF_extended_context(test_ctx)
test_state = allocate_CS_4c_KU_SCF_state(test_ctx)

initialize_CS_4c_KU_SCF_extended_context(test_ctx, test_const)
initialize_CS_4c_KU_SCF_P_and_E(test_ctx, test_state)
initialize_CS_4c_KU_SCF_state_variable(test_const, test_state)

In [9]:
_kuscf_results = _kuscf_kernel(test_ctx)

-------------------------------------------------------------------------------------------------------------------------------------
|   Iter   |                    E_iter                     |                    Delta_e                    |       norm(e_i)        |
-------------------------------------------------------------------------------------------------------------------------------------
    1     -4872688.1433527227491140+0.0000000000000000j     -4872688.1433527227491140+0.0000000000000000j     1.1283E+07
    2            0.3329562595477592-0.0000000000000000j      4872688.4763089818879962-0.0000000000000000j     1.6203E+01
    3           -2.7888741233004799-0.0000000000000000j           -3.1218303828482390+0.0000000000000000j     1.2334E+00
    4           -2.8608343629042920-0.0000000000000000j           -0.0719602396038121+0.0000000000000000j     6.1276E-02
    5           -2.8612778664554379+0.0000000000000000j           -0.0004435035511459+0.0000000000000000j     5.81

Which seems to work with He. 

---
